In [1]:
import numpy as np
import pandas as pd
import polyline
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

In [2]:
import os
print(os.getcwd())
print(os.listdir())

C:\Users\DELL\ML\SupplyChainMapping
['.ipynb_checkpoints', 'IsolateForest.ipynb', 'TripDataGeneration.py', 'trip_data (1).csv']


In [3]:
df = pd.read_csv("trip_data (1).csv")

print(df.head())

   route_id  origin_lat  origin_lon   dest_lat   dest_lon  distance_m  \
0         1   28.505403   76.925911  28.895684  77.407156     73457.8   
1         2   28.821007   77.095695  28.712488  76.867139     40679.1   
2         4   28.756587   76.961965  28.466256  77.333092     54989.7   
3         5   28.588656   76.985224  28.703552  77.026946     20786.2   
4         6   28.732450   77.413181  28.690516  77.297214      5543.0   

   duration_s                                           polyline  
0      5417.9  y`~lDu{ntMo@b@CHAH?hA@hCCjAArBAd@@f@E~GClLIrd@...  
1      2659.2  m`|nDuppuMjMmDH^qNxDqNvDoI|BUp@aArCDJ@FCPGJIDK...  
2      3749.5  euonDoyvtMJ_E?g@Au@?EAc@@e@?_@a@?@aA?iA?cABaA?...  
3      1320.8  apnmDsa{tM?pBCdEErG?v@Ef@i@?gAAyA?aAAuGBmCKuAA...  
4       578.2  a`gnDax~vMLh@Lh@Tx@@F@f@?F@\@HJpA@FLv@BL@j@BFB...  


In [4]:
df["coordinates"] = df["polyline"].apply(polyline.decode)
print(df.head())

   route_id  origin_lat  origin_lon   dest_lat   dest_lon  distance_m  \
0         1   28.505403   76.925911  28.895684  77.407156     73457.8   
1         2   28.821007   77.095695  28.712488  76.867139     40679.1   
2         4   28.756587   76.961965  28.466256  77.333092     54989.7   
3         5   28.588656   76.985224  28.703552  77.026946     20786.2   
4         6   28.732450   77.413181  28.690516  77.297214      5543.0   

   duration_s                                           polyline  \
0      5417.9  y`~lDu{ntMo@b@CHAH?hA@hCCjAArBAd@@f@E~GClLIrd@...   
1      2659.2  m`|nDuppuMjMmDH^qNxDqNvDoI|BUp@aArCDJ@FCPGJIDK...   
2      3749.5  euonDoyvtMJ_E?g@Au@?EAc@@e@?_@a@?@aA?iA?cABaA?...   
3      1320.8  apnmDsa{tM?pBCdEErG?v@Ef@i@?gAAyA?aAAuGBmCKuAA...   
4       578.2  a`gnDax~vMLh@Lh@Tx@@F@f@?F@\@HJpA@FLv@BL@j@BFB...   

                                         coordinates  
0  [(28.50333, 76.92235), (28.50357, 76.92217), (...  
1  [(28.82071, 77.09467), (28.81841, 77.09

In [5]:
from geopy.distance import geodesic

def compute_length(coords):
    total = 0
    for i in range(len(coords)-1):
        total += geodesic(coords[i], coords[i+1]).meters
    return total

df["route_length_m"] = df["coordinates"].apply(compute_length)
print(df.head())

   route_id  origin_lat  origin_lon   dest_lat   dest_lon  distance_m  \
0         1   28.505403   76.925911  28.895684  77.407156     73457.8   
1         2   28.821007   77.095695  28.712488  76.867139     40679.1   
2         4   28.756587   76.961965  28.466256  77.333092     54989.7   
3         5   28.588656   76.985224  28.703552  77.026946     20786.2   
4         6   28.732450   77.413181  28.690516  77.297214      5543.0   

   duration_s                                           polyline  \
0      5417.9  y`~lDu{ntMo@b@CHAH?hA@hCCjAArBAd@@f@E~GClLIrd@...   
1      2659.2  m`|nDuppuMjMmDH^qNxDqNvDoI|BUp@aArCDJ@FCPGJIDK...   
2      3749.5  euonDoyvtMJ_E?g@Au@?EAc@@e@?_@a@?@aA?iA?cABaA?...   
3      1320.8  apnmDsa{tM?pBCdEErG?v@Ef@i@?gAAyA?aAAuGBmCKuAA...   
4       578.2  a`gnDax~vMLh@Lh@Tx@@F@f@?F@\@HJpA@FLv@BL@j@BFB...   

                                         coordinates  route_length_m  
0  [(28.50333, 76.92235), (28.50357, 76.92217), (...    73466.283382  
1  [(28.82

In [6]:
#straight line distance
# geodesic(coords[0], coords[-1]).meters
# df["straight_dist_m"] = df["coordinates"].apply(straight_distance)

# from geopy.distance import geodesic


In [7]:
from geopy.distance import geodesic

straight_distances = []

for index, row in df.iterrows():
    start = (row["origin_lat"], row["origin_lon"])
    end   = (row["dest_lat"], row["dest_lon"])
    
    distance = geodesic(start, end).meters
    straight_distances.append(distance)

df["straight_dist_m"] = straight_distances

print(df.head())


   route_id  origin_lat  origin_lon   dest_lat   dest_lon  distance_m  \
0         1   28.505403   76.925911  28.895684  77.407156     73457.8   
1         2   28.821007   77.095695  28.712488  76.867139     40679.1   
2         4   28.756587   76.961965  28.466256  77.333092     54989.7   
3         5   28.588656   76.985224  28.703552  77.026946     20786.2   
4         6   28.732450   77.413181  28.690516  77.297214      5543.0   

   duration_s                                           polyline  \
0      5417.9  y`~lDu{ntMo@b@CHAH?hA@hCCjAArBAd@@f@E~GClLIrd@...   
1      2659.2  m`|nDuppuMjMmDH^qNxDqNvDoI|BUp@aArCDJ@FCPGJIDK...   
2      3749.5  euonDoyvtMJ_E?g@Au@?EAc@@e@?_@a@?@aA?iA?cABaA?...   
3      1320.8  apnmDsa{tM?pBCdEErG?v@Ef@i@?gAAyA?aAAuGBmCKuAA...   
4       578.2  a`gnDax~vMLh@Lh@Tx@@F@f@?F@\@HJpA@FLv@BL@j@BFB...   

                                         coordinates  route_length_m  \
0  [(28.50333, 76.92235), (28.50357, 76.92217), (...    73466.283382   
1  [(28.

In [8]:
#route efficiancy
df["efficiency_ratio"] = df["straight_dist_m"] / df["route_length_m"]
print(df.head())

   route_id  origin_lat  origin_lon   dest_lat   dest_lon  distance_m  \
0         1   28.505403   76.925911  28.895684  77.407156     73457.8   
1         2   28.821007   77.095695  28.712488  76.867139     40679.1   
2         4   28.756587   76.961965  28.466256  77.333092     54989.7   
3         5   28.588656   76.985224  28.703552  77.026946     20786.2   
4         6   28.732450   77.413181  28.690516  77.297214      5543.0   

   duration_s                                           polyline  \
0      5417.9  y`~lDu{ntMo@b@CHAH?hA@hCCjAArBAd@@f@E~GClLIrd@...   
1      2659.2  m`|nDuppuMjMmDH^qNxDqNvDoI|BUp@aArCDJ@FCPGJIDK...   
2      3749.5  euonDoyvtMJ_E?g@Au@?EAc@@e@?_@a@?@aA?iA?cABaA?...   
3      1320.8  apnmDsa{tM?pBCdEErG?v@Ef@i@?gAAyA?aAAuGBmCKuAA...   
4       578.2  a`gnDax~vMLh@Lh@Tx@@F@f@?F@\@HJpA@FLv@BL@j@BFB...   

                                         coordinates  route_length_m  \
0  [(28.50333, 76.92235), (28.50357, 76.92217), (...    73466.283382   
1  [(28.

In [9]:
#average speed
df["avg_speed_mps"] = df["route_length_m"] / df["duration_s"]
print(df.head())

   route_id  origin_lat  origin_lon   dest_lat   dest_lon  distance_m  \
0         1   28.505403   76.925911  28.895684  77.407156     73457.8   
1         2   28.821007   77.095695  28.712488  76.867139     40679.1   
2         4   28.756587   76.961965  28.466256  77.333092     54989.7   
3         5   28.588656   76.985224  28.703552  77.026946     20786.2   
4         6   28.732450   77.413181  28.690516  77.297214      5543.0   

   duration_s                                           polyline  \
0      5417.9  y`~lDu{ntMo@b@CHAH?hA@hCCjAArBAd@@f@E~GClLIrd@...   
1      2659.2  m`|nDuppuMjMmDH^qNxDqNvDoI|BUp@aArCDJ@FCPGJIDK...   
2      3749.5  euonDoyvtMJ_E?g@Au@?EAc@@e@?_@a@?@aA?iA?cABaA?...   
3      1320.8  apnmDsa{tM?pBCdEErG?v@Ef@i@?gAAyA?aAAuGBmCKuAA...   
4       578.2  a`gnDax~vMLh@Lh@Tx@@F@f@?F@\@HJpA@FLv@BL@j@BFB...   

                                         coordinates  route_length_m  \
0  [(28.50333, 76.92235), (28.50357, 76.92217), (...    73466.283382   
1  [(28.

In [11]:
#feature matrix
features = df[[
    "avg_speed_mps",
    "route_length_m",
    "straight_dist_m",
    "efficiency_ratio"
]]

In [14]:
from sklearn.ensemble import IsolationForest

model = IsolationForest(
    contamination=0.05,
    random_state=42
)

model.fit(features)

IsolationForest(contamination=0.05, random_state=42)

In [16]:
df["anomaly"] = model.predict(features)
df["anomaly_score"] = model.decision_function(features)
print(df.head())

   route_id  origin_lat  origin_lon   dest_lat   dest_lon  distance_m  \
0         1   28.505403   76.925911  28.895684  77.407156     73457.8   
1         2   28.821007   77.095695  28.712488  76.867139     40679.1   
2         4   28.756587   76.961965  28.466256  77.333092     54989.7   
3         5   28.588656   76.985224  28.703552  77.026946     20786.2   
4         6   28.732450   77.413181  28.690516  77.297214      5543.0   

   duration_s                                           polyline  \
0      5417.9  y`~lDu{ntMo@b@CHAH?hA@hCCjAArBAd@@f@E~GClLIrd@...   
1      2659.2  m`|nDuppuMjMmDH^qNxDqNvDoI|BUp@aArCDJ@FCPGJIDK...   
2      3749.5  euonDoyvtMJ_E?g@Au@?EAc@@e@?_@a@?@aA?iA?cABaA?...   
3      1320.8  apnmDsa{tM?pBCdEErG?v@Ef@i@?gAAyA?aAAuGBmCKuAA...   
4       578.2  a`gnDax~vMLh@Lh@Tx@@F@f@?F@\@HJpA@FLv@BL@j@BFB...   

                                         coordinates  route_length_m  \
0  [(28.50333, 76.92235), (28.50357, 76.92217), (...    73466.283382   
1  [(28.